In [11]:
%%writefile practice91.cpp
// Задание 1: Распределённое вычисление среднего значения и стандартного отклонения
// 1. Создайте массив случайных чисел на процессе с "rank = 0". Размер массива
// 2. Разделите массив между всеми процессами с помощью функции "MPI_Scatterv" (учитывая, что массив может не делиться нацело между процессами).
// 3. Каждый процесс вычисляет:
   // - Сумму элементов своей части массива.
   // - Сумму квадратов элементов своей части массива.
// 4. Соберите локальные суммы на процессе с "rank = 0" с помощью функции "MPI_Reduce".
// 5. На основе собранных данных вычислите:
   // - Среднее значение массива.
   // - Стандартное отклонение, формула:
// 6. Выведите результаты на экран.
#include <mpi.h>        // Библиотека MPI для распределённых вычислений
#include <iostream>     // Для вывода на экран
#include <vector>       // Для динамических массивов (std::vector)
#include <cmath>        // Для sqrt
#include <cstdlib>      // Для rand()

int main(int argc, char* argv[]) {
    MPI_Init(&argc, &argv);                // Инициализация MPI
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);  // Получаем уникальный номер текущего процесса (rank)
    MPI_Comm_size(MPI_COMM_WORLD, &size);  // Получаем общее количество процессов (size)

    const int N = 1000000;                 // Размер глобального массива
    std::vector<double> data;              // Глобальный массив данных на rank 0

    double start_time = MPI_Wtime();       // Замер времени начала выполнения программы

    if (rank == 0) {                       // Только главный процесс создаёт массив
        data.resize(N);                    // Выделяем память под массив из N элементов
        for (int i = 0; i < N; i++)       // Заполняем массив случайными числами
            data[i] = rand() / (double)RAND_MAX; // Случайные числа в диапазоне [0,1)
    }

    std::vector<int> counts(size);        // Массив, сколько элементов получит каждый процесс
    std::vector<int> displs(size);        // Массив смещений (откуда брать элементы для каждого процесса)

    int rem = N % size;                   // Остаток при делении массива на число процессов
    int sum = 0;                           // Сумма для вычисления смещений
    for (int i = 0; i < size; i++) {
        counts[i] = N / size;             // Базовое количество элементов на процесс
        if (i < rem) counts[i]++;         // Распределяем остаток: первые rem процессов получают +1 элемент
        displs[i] = sum;                  // Смещение для текущего процесса
        sum += counts[i];                 // Обновляем сумму для следующего процесса
    }

    std::vector<double> local_data(counts[rank]);  // Локальный массив для процесса rank

    // Распределяем части массива data между процессами
    MPI_Scatterv(data.data(), counts.data(), displs.data(),
                 MPI_DOUBLE, local_data.data(), counts[rank],
                 MPI_DOUBLE, 0, MPI_COMM_WORLD);

    double local_sum = 0.0, local_sq_sum = 0.0;  // Локальные сумма и сумма квадратов
    for (double x : local_data) {                // Перебираем элементы локальной части массива
        local_sum += x;                          // Локальная сумма
        local_sq_sum += x * x;                   // Локальная сумма квадратов
    }

    double global_sum = 0.0, global_sq_sum = 0.0;  // Глобальные сумма и сумма квадратов
    MPI_Reduce(&local_sum, &global_sum, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);       // Суммируем локальные суммы
    MPI_Reduce(&local_sq_sum, &global_sq_sum, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD); // Суммируем локальные суммы квадратов

    if (rank == 0) {                                // На главном процессе вычисляем среднее и стандартное отклонение
        double mean = global_sum / N;              // Среднее значение
        double stddev = std::sqrt(global_sq_sum / N - mean * mean); // Стандартное отклонение

        double end_time = MPI_Wtime();             // Замер времени конца выполнения
        std::cout << "Среднее значение массива = " << mean << "\n";
        std::cout << "Стандартное отклонение = " << stddev << "\n";
        std::cout << "Время выполнения = " << end_time - start_time << " с\n"; // Вывод времени выполнения
    }

    MPI_Finalize();                        // Завершаем работу MPI
    return 0;                               // Завершение программы
}



Overwriting practice91.cpp


In [20]:
# Компиляция
!mpic++ practice91.cpp -o practice91
# mpirun — утилита, которая запускает MPI-программу на указанном числе процессов
# --allow-run-as-root — разрешает запуск MPI от root (Colab запускает ядра как root)
# --oversubscribe — игнорирует количество доступных виртуальных CPU, позволяя запускать больше процессов, чем физически есть
# -np 2 — число процессов MPI (2 процесса)

# Запуск с 2 процессами
!mpirun --allow-run-as-root --oversubscribe -np 2 ./practice91

# Запуск с 4 процессами
!mpirun --allow-run-as-root --oversubscribe -np 4 ./practice91

# Запуск с 8 процессами
!mpirun --allow-run-as-root --oversubscribe -np 8 ./practice91

Среднее значение массива = 0.500007
Стандартное отклонение = 0.28862
Время выполнения = 0.0305639 с
Среднее значение массива = 0.500007
Стандартное отклонение = 0.28862
Время выполнения = 0.0351235 с
Среднее значение массива = 0.500007
Стандартное отклонение = 0.28862
Время выполнения = 0.110057 с
